[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/08_soft_distance.ipynb)

# Soft topics, soft lenses, and earth mover's distance

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-18

**Abstract**: Soft-throughout Address keeps phrase→topic responsibilities from a mixture of
PPCA (notebook 03), then softens framing lenses with a second mixture of PPCAs
over topic means. Earth mover's distance appears twice — once when choosing
lens $\sigma^2$ across independent LLM runs, and once when comparing documents
as distributions over topics (preferably soft histograms). Two noise floors stay
independent and fixed per run. This notebook teaches both geometries offline.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

Framing lenses — LLM labels such as economic, social, or environmental — are
useful for interpretation but brittle if treated as hard membership.
Responsibilities give every *topic* a distribution over lenses rather than a
single label. Separately, documents can be compared by transport distance on
their **soft** topic-share profiles without using lens names at all. Same
Kantorovich programme; different histograms and ground costs.


## Two $\sigma^2$ floors (CIP-0017)

| Floor | Role |
|-------|------|
| Topic $\sigma^2_{\mathrm{topic}}$ | Fixed in `mppca_fit` (notebook 03) |
| Lens $\sigma^2_{\mathrm{lens}}$ | Shared when scoring lens runs; $\sigma^{2\star}$ by EMD below, then fixed |

Do not re-fit a different $\sigma^2$ per lens replicate or per document.


In [ ]:
# Optional transport dependency (soft lenses + document EMD).
import importlib.util, subprocess, sys
if importlib.util.find_spec("ot") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pot"])
print("POT available")


# Soft topics → soft document histograms


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from multilogue.mppca import mppca_fit, DEFAULT_TOPIC_SIGMA2
from multilogue import distance as dd
from multilogue import viz


In [ ]:
rng = np.random.default_rng(0)
# Planted soft geometry: three subspace topics, then soft document histograms
n_per, d, k = 40, 12, 3
X = []
doc_ids = []
for j in range(k):
    mean = np.zeros(d)
    mean[2 * j] = 2.5
    cov = np.eye(d) * 0.08
    cov[2 * j, 2 * j] = 0.6
    Xj = rng.multivariate_normal(mean, cov, size=n_per)
    X.append(Xj)
    # two docs per topic component
    doc_ids += [f"d{j}a"] * (n_per // 2) + [f"d{j}b"] * (n_per - n_per // 2)
X = np.vstack(X)
fit = mppca_fit(X, k=k, sigma2=DEFAULT_TOPIC_SIGMA2, seed=0)
R = fit["responsibilities"]
P_soft, docs_soft, mass = dd.document_soft_topic_distributions(
    doc_ids, R, min_mass=5.0,
)
print(f"soft docs kept: {docs_soft}; mass={mass.to_dict()}")
print("P_soft rows sum:", P_soft.sum(axis=1))


`document_soft_topic_distributions` sums phrase responsibilities per document
and normalises — the CIP-0017 default for document EMD. The hard-count helper
`document_cluster_distributions` remains a sensitivity view.


# Soft lenses


## Mixture responsibilities

Hard clustering gives every point a single label: point $i$ belongs to component
$c(i)$ and to no other. A *mixture* model treats the component index as a latent
variable and returns a full posterior. Write $\pi_c = p(c)$ for the prior weight
of component $c$, and $p(\mathbf{y}_i\mid c)$ for that component's density.
Bayes' rule yields the *responsibility*
$$
r_{ic}
=
p(c\mid\mathbf{y}_i)
=
\frac{
\pi_c\, p(\mathbf{y}_i\mid c)
}{
\sum_{c'} \pi_{c'}\, p(\mathbf{y}_i\mid c')
}.
$$
The row $(r_{i1},\ldots,r_{iC})$ is a discrete distribution over components for
point $i$. Hard assignment is the special case that replaces this row by a
one-hot vector at $\arg\max_c r_{ic}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from IPython.display import display


In [ ]:
rng = np.random.default_rng(0)
means = np.array([[-1.2, 0.0], [1.2, 0.3]])
sigma2_sharp, sigma2_soft = 0.35, 2.0
priors = np.array([0.5, 0.5])
probes = np.array([
    [-1.2, 0.0], [-0.4, 0.1], [0.0, 0.15], [0.4, 0.2], [1.2, 0.3],
])

def responsibilities(X, means, sigma2, priors):
    log_lik = np.stack([
        -0.5 * np.sum((X - m) ** 2, axis=1) / sigma2 for m in means
    ], axis=1)
    log_post = log_lik + np.log(priors)[None, :]
    log_post -= log_post.max(axis=1, keepdims=True)
    post = np.exp(log_post)
    return post / post.sum(axis=1, keepdims=True)

cloud = np.vstack([
    rng.normal(means[0], np.sqrt(sigma2_sharp), size=(40, 2)),
    rng.normal(means[1], np.sqrt(sigma2_sharp), size=(40, 2)),
])
colours = ["#4C72B0", "#DD8452"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, sigma2, title in [
    (axes[0], sigma2_sharp, rf"sharp ($\sigma^2={sigma2_sharp:g}$)"),
    (axes[1], sigma2_soft, rf"soft ($\sigma^2={sigma2_soft:g}$)"),
]:
    R = responsibilities(probes, means, sigma2, priors)
    ax.scatter(cloud[:, 0], cloud[:, 1], s=12, c="#bbbbbb", alpha=0.7, zorder=1)
    for c, m in enumerate(means):
        ax.plot(*m, "o", color=colours[c], markersize=10, zorder=3)
        ell = Ellipse(
            m, 2 * np.sqrt(sigma2), 2 * np.sqrt(sigma2),
            fill=False, ec=colours[c], lw=1.5, ls="--", zorder=2,
        )
        ax.add_patch(ell)
    bar_w, bar_h = 0.35, 0.9
    for x, r in zip(probes, R):
        y0 = x[1] + 0.55
        ax.add_patch(plt.Rectangle(
            (x[0] - bar_w / 2, y0), bar_w * r[0], bar_h,
            color=colours[0], zorder=4, clip_on=False,
        ))
        ax.add_patch(plt.Rectangle(
            (x[0] - bar_w / 2 + bar_w * r[0], y0), bar_w * r[1], bar_h,
            color=colours[1], zorder=4, clip_on=False,
        ))
        ax.plot(x[0], x[1], "k.", markersize=8, zorder=5)
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(-2.0, 2.8)
    ax.set_aspect("equal")
    ax.set_title(title)
fig.tight_layout()
fig.savefig(FIG / "08_mixture_responsibilities.png", dpi=120, bbox_inches="tight")
plt.show()


The figure places five probe points between two isotropic Gaussian components.
Stacked bars above each probe show the responsibility vector
$(r_{i1}, r_{i2})$. When $\sigma^2$ is small, probes near a mean are almost
one-hot — soft assignment collapses toward hard labels. When $\sigma^2$ is
large, the same probes carry mass on both components. Only the component
densities — and therefore the posteriors — have become flatter.


## Choosing $\sigma^2$ by cross-run EMD

Independent LLM lens runs disagree. Multilogue fits a PPCA Gaussian per lens
over member cluster centroids, recomputes mixture weights for each candidate
$\sigma^2$, and measures mean earth mover's distance between runs (ground cost =
cosine distance between lens means). We pick $\sigma^{2\star}$ that minimises
that EMD among values whose mean *effective support* lies in a meaningful-soft
band — so we do not collapse to the near-uniform high-$\sigma^2$ solution where
group contrasts vanish.


In [ ]:
from multilogue import soft as ms


In [ ]:
k_lens, dim, n_lenses, n_runs = 24, 12, 6, 3
centroids = rng.normal(size=(k_lens, dim))
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
mappings = []
for _ in range(n_runs):
    perm = rng.permutation(k_lens)
    mappings.append({
        f"L{j}": {"cluster_ids": perm[j::n_lenses].tolist()} for j in range(n_lenses)
    })

sweep = ms.sigma2_sweep_select(centroids, mappings)
curve = pd.DataFrame(sweep["curve"])
print(f"Selected lens σ²*: {sweep['sigma2_best']:.4g} ({sweep['selection']})")
print(f"Mean EMD at σ²*: {sweep['emd_best']:.4f}")
print(f"Mean eff. support: {sweep['eff_support_best']:.2f}")
display(curve[["sigma2", "mean_emd", "mean_eff_support"]].head(8))

W, names = ms.compute_mixture_weights(centroids, mappings[0], sweep["sigma2_best"])
print("Mixture weights shape:", W.shape, "lenses:", names)


In [ ]:
viz.sigma2_stability_curve(curve, path=FIG / "08_sigma2_curve.png")
viz.mixture_weight_heatmap(W, names, path=FIG / "08_mixture_weights.png")


The stability curve shows mean cross-run EMD against $\sigma^2$; the selected
value is marked when the band is non-empty. The heatmap shows soft mixture
weights for one run at $\sigma^{2\star}$ — rows are clusters, columns are lenses.


# Comparing distributions


## Earth mover's distance

The classical metaphor is dirt and work. One histogram is a pile of earth; the
other is the target landscape. A plan specifies how much mass travels from each
source bin to each destination bin. The total work is mass times ground cost,
summed over shipments. The EMD is the work of the *cheapest* such plan.

If the ground cost is a metric on the bins — for example cosine distance between
cluster centroids — the EMD inherits that geometry: documents whose topic mass
sits on nearby centroids look close; documents that raise unrelated topics look
far.


## Discrete optimal transport

Write source and target histograms $a\in\mathbb{R}^{m}$, $b\in\mathbb{R}^{n}$
with $\sum_i a_i=\sum_j b_j=1$. A coupling $\Pi\ge 0$ has row sums $a$ and
column sums $b$. Given ground cost $C$,
$$
\mathrm{EMD}(a,b;C)
=
\min_{\Pi:\,\Pi\mathbf{1}=a,\,\Pi^{\top}\mathbf{1}=b}
\langle\Pi,C\rangle.
$$
This is Kantorovich's formulation. POT's `ot.emd` / `ot.emd2` return the optimal
cost and a coupling.


In [ ]:
# Soft-histogram document EMD (CIP-0017 path)
prof_a = np.array([0.55, 0.35, 0.10])
prof_b = np.array([0.10, 0.35, 0.55])
rows_R = []
rows_doc = []
for i in range(8):
    for _ in range(40):
        rows_R.append(prof_a)
        rows_doc.append(f"a{i}")
    for _ in range(40):
        rows_R.append(prof_b)
        rows_doc.append(f"b{i}")
R_docs = np.asarray(rows_R, float)
# Add a little noise then renormalise so rows stay soft
R_docs = np.clip(R_docs + rng.normal(0, 0.02, size=R_docs.shape), 1e-6, None)
R_docs /= R_docs.sum(axis=1, keepdims=True)
P, docs, mass = dd.document_soft_topic_distributions(rows_doc, R_docs, min_mass=20.0)
means3 = fit["means"]
M = dd.cosine_ground_cost(means3)
D = dd.emd_distance_matrix(P, M)
groups = np.array(["A" if d.startswith("a") else "B" for d in docs])
pm = dd.permanova(D, groups, n_perm=199, seed=0)
mds = dd.classical_mds(D, 2)
print(f"Soft docs: {len(docs)}; PERMANOVA R²={pm['R2']:.3f}, p={pm['p_value']:.4f}")


In [ ]:
coords = mds["coords"]
fig, ax = plt.subplots(figsize=(5, 4))
for g, marker in (("A", "o"), ("B", "s")):
    m = groups == g
    ax.scatter(coords[m, 0], coords[m, 1], label=g, marker=marker)
ax.set_xlabel("MDS 1")
ax.set_ylabel("MDS 2")
ax.set_title("Soft-histogram document EMD → classical MDS")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "08_doc_mds.png", dpi=120)
plt.show()


Two planted groups with different soft topic profiles separate under EMD → MDS.
PERMANOVA tests whether group locations differ in the transport geometry —
answering whether groups differ in *what* they raise, without using lens names.


# Outlook

What remains in consumer packages is study-specific: corpora, prompts, and the
precise definition of headline $\ell$. Soft mixture weights also feed
`soft_group_compare` for document-weighted soft contrasts. Notebook 09 puts
format as the group column and a format-gap $\ell$ through the full stack.


## References

Tipping, M.E., Bishop, C.M., 1999. Probabilistic principal component analysis.
Journal of the Royal Statistical Society, B 61, 611–622.
